# Company N Genre & Language Classification## Data Source- **Input**: company_n.csv (transaction log with dates)- **Data Type**: Individual checkout transactions (grouped by title for frequency count)- **Processing**: Groups transactions by book title, detects language, assigns genres, generates enriched output with analyticsSee [DATA_SEMANTICS.md](../../Data/DATA_SEMANTICS.md) for details on data interpretation across all companies.

In [3]:
import osimport jsonimport reimport unicodedatafrom typing import List, Tuple, Dict, Optionalfrom collections import defaultdictfrom functools import lru_cacheimport concurrent.futuresimport pandas as pdimport langidimport requestsfrom requests.adapters import HTTPAdapterfrom urllib3.util.retry import Retryimport warningswarnings.filterwarnings("ignore", category=Warning)# CONFIGCOMPANY_N_FILE = "company_n.csv"OUTPUT_COMPANY_N = "company_n_genres_output.csv"TOP_K_GENRES = 3OPENLIBRARY_CACHE_FILE = "company_n_openlibrary_cache.json"# LANGUAGELANGUAGE_NAMES = {    "en": "English",    "fr": "French",    "es": "Spanish",    "de": "German",    "it": "Italian",    "pt": "Portuguese",    "ru": "Russian",    "hy": "Armenian",    "tr": "Turkish",    "ar": "Arabic",    "zh": "Chinese",    "ja": "Japanese",    "ko": "Korean",}EN_HINT_WORDS = frozenset({"the","a","an","of","and","to","in","for","with","on"})# GENRE MAPPING (same list as other pipelines)GENRES = [    "Fantasy","Science Fiction","Romance","Mystery","Thriller",    "Historical Fiction","Nonfiction","Biography","Young Adult","Horror",]GENRE_KEYWORDS = {    "Fantasy": frozenset(["fantasy","magic","dragon","myth","middle earth"]),    "Science Fiction": frozenset(["science fiction","sci-fi","space","alien","dystop"]),    "Romance": frozenset(["romance","love"]),    "Mystery": frozenset(["mystery","detective","crime"]),    "Thriller": frozenset(["thriller","suspense"]),    "Horror": frozenset(["horror","ghost","haunted"]),    "Biography": frozenset(["biography","autobiography","memoir"]),    "Nonfiction": frozenset(["nonfiction","history","business","psychology","self-help"]),    "Historical Fiction": frozenset(["historical fiction"]),    "Young Adult": frozenset(["young adult","ya"]),}WHITESPACE_RE = re.compile(r"\s+")@lru_cache(maxsize=1024)def detect_language(title: str, min_confidence_latin: float = 0.35) -> Tuple[str,str,float]:    text = (title or "").strip()    if not text:        return "unknown","Unknown",0.0    for ch in text:        cp = ord(ch)        if 0x0530 <= cp <= 0x058F: return "hy","Armenian",1.0        if 0x0400 <= cp <= 0x04FF: return "cyr","Cyrillic",1.0        if 0x0600 <= cp <= 0x06FF: return "ar","Arabic",1.0        if 0x0590 <= cp <= 0x05FF: return "he","Hebrew",1.0        if 0x0370 <= cp <= 0x03FF: return "el","Greek",1.0        if 0x4E00 <= cp <= 0x9FFF: return "cjk","CJK",1.0    normalized = unicodedata.normalize("NFKD", text)    normalized = "".join(c for c in normalized if not unicodedata.combining(c))    words = {w.lower() for w in normalized.replace("'"," ").split()}    if words & EN_HINT_WORDS: return "en","English",0.99    code,conf = langid.classify(normalized)    if conf >= min_confidence_latin: return code, LANGUAGE_NAMES.get(code,code), float(conf)    return "latin","Latin (Unknown language)",float(conf)# OPENLIBRARY CACHE + SESSIONOPENLIBRARY_CACHE: Dict[str,List[str]] = {}def _load_cache():    try:        if os.path.exists(OPENLIBRARY_CACHE_FILE):            with open(OPENLIBRARY_CACHE_FILE,'r',encoding='utf-8') as f:                data = json.load(f)                if isinstance(data,dict): OPENLIBRARY_CACHE.update(data)    except Exception:        passdef _save_cache():    try:        with open(OPENLIBRARY_CACHE_FILE,'w',encoding='utf-8') as f:            json.dump(OPENLIBRARY_CACHE,f,ensure_ascii=False)    except Exception:        pass_session: Optional[requests.Session] = Nonedef _session_with_retries():    global _session    if _session is None:        s = requests.Session()        retries = Retry(total=3, backoff_factor=0.6, status_forcelist=(500,502,503,504))        s.mount('https://', HTTPAdapter(max_retries=retries))        _session = s    return _session_load_cache()def openlibrary_get_subjects(title: str) -> List[str]:    title = (title or '').strip()    if not title: return []    if title in OPENLIBRARY_CACHE: return OPENLIBRARY_CACHE[title]    session = _session_with_retries()    try:        r = session.get('https://openlibrary.org/search.json', params={'title': title}, timeout=8)        r.raise_for_status()        data = r.json()        docs = data.get('docs',[])        if docs and docs[0].get('subject'):            subjects = docs[0]['subject']            OPENLIBRARY_CACHE[title] = subjects            return subjects        if docs:            work_key = docs[0].get('key')            if work_key:                w = session.get(f'https://openlibrary.org{work_key}.json', timeout=8)                w.raise_for_status()                subjects = w.json().get('subjects',[]) or []                OPENLIBRARY_CACHE[title] = subjects                return subjects    except Exception:        OPENLIBRARY_CACHE[title] = []        return []    OPENLIBRARY_CACHE[title] = []    return []def normalize_text(s: str) -> str:    return WHITESPACE_RE.sub(' ', (s or '').lower().strip())def map_subjects_to_genres(subjects: List[str], top_k: int) -> List[str]:    if not subjects: return []    text = ' | '.join(normalize_text(x) for x in subjects)    found = []    for genre, keys in GENRE_KEYWORDS.items():        if any(k in text for k in keys):            found.append(genre)            if len(found) >= top_k: break    return found[:top_k]# AI FALLBACK (lazy + batched)_classifier = None_classifier_model = 'valhalla/distilbart-mnli-12-1'def get_classifier():    global _classifier    if _classifier is None:        try:            import torch            from transformers import pipeline, AutoConfig            device = 0 if torch.cuda.is_available() else -1            cfg = AutoConfig.from_pretrained(_classifier_model)            cfg.tie_word_embeddings = False            _classifier = pipeline('zero-shot-classification', model=_classifier_model, config=cfg, device=device)        except Exception:            _classifier = None    return _classifierdef get_genres_for_titles(titles: List[str], top_k: int) -> List[List[str]]:    mapped = [map_subjects_to_genres(openlibrary_get_subjects(t), top_k) for t in titles]    missing = [i for i,m in enumerate(mapped) if not m]    if not missing: return mapped    clf = get_classifier()    if clf is None: return mapped    batch = 16    for i in range(0, len(missing), batch):        batch_idx = missing[i:i+batch]        batch_titles = [titles[j] for j in batch_idx]        try:            res = clf(batch_titles, candidate_labels=GENRES, hypothesis_template='This book is a {} book.')        except Exception:            res = []        if isinstance(res, dict): res = [res]        for j,r in enumerate(res):            labels = r.get('labels',[])[:top_k]            mapped[batch_idx[j]] = labels    return mapped# UTIL: clean titledef clean_title(raw: str) -> str:    s = '' if raw is None else str(raw)    s = unicodedata.normalize('NFKC', s)    s = s.strip()    s = re.sub(r'\s+',' ', s)    s = s.lower()    s = re.sub(r'[^\w\s]',' ', s)    s = re.sub(r'\s+',' ', s).strip()    return s# CORE PIPELINEdef run_pipeline(df: pd.DataFrame, output_file: str):    titles = df['Title'].astype(str).tolist()    numbers = df['Number'].astype(float).tolist()    langs = [detect_language(t)[1] for t in titles]    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:        subjects_list = list(ex.map(openlibrary_get_subjects, titles))    mapped = [map_subjects_to_genres(s, TOP_K_GENRES) for s in subjects_list]    missing = [i for i,m in enumerate(mapped) if not m]    if missing:        to_classify = [titles[i] for i in missing]        classified = get_genres_for_titles(to_classify, TOP_K_GENRES)        for idx, labels in zip(missing, classified): mapped[idx] = labels    rows = []    genre_title_count = defaultdict(int)    genre_number_sum = defaultdict(float)    for title,num,lang,genres in zip(titles,numbers,langs,mapped):        genres = genres or []        for g in genres:            genre_title_count[g] += 1            genre_number_sum[g] += num / max(len(genres),1)        rows.append({'Title': title, 'Number': num, 'Language': lang, 'Genres': ', '.join(genres)})    result_df = pd.DataFrame(rows)    top_titles = sorted(genre_title_count.items(), key=lambda x: x[1], reverse=True)[:5]    top_numbers = sorted(genre_number_sum.items(), key=lambda x: x[1], reverse=True)[:5]    summary_text = (f"Top genres by title count: {top_titles}. "                    f"Top genres by total Number: {[(g,int(v)) for g,v in top_numbers]}.")    output_data = rows + [{'Title':'','Number':'','Language':'','Genres':''}, {'Title':'SUMMARY','Number':int(result_df['Number'].sum()),'Language':'','Genres':summary_text}]    final_df = pd.DataFrame(output_data)    final_df.to_csv(output_file, index=False, encoding='utf-8')    _save_cache()    print(f" Saved {output_file} ({len(rows)} titles processed)")# DATA PREPdef prepare_company_n(path: str) -> pd.DataFrame:    df = pd.read_csv(path, sep=None, engine='python', encoding='utf-8-sig', on_bad_lines='skip')    if 'Book Names' not in df.columns:        raise ValueError('expected column "Book Names" in input CSV')    df['Book Names'] = df['Book Names'].astype(str)    df['CleanTitle'] = df['Book Names'].apply(clean_title)    df = df[df['CleanTitle'] != '']    grouped = df.groupby('CleanTitle', as_index=False).size().rename(columns={'CleanTitle':'Title','size':'Number'})    return grouped# MAINdef main():    df_in = prepare_company_n(COMPANY_N_FILE)    run_pipeline(df_in, OUTPUT_COMPANY_N)if __name__ == '__main__':    main()

Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warningThe tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning

 Saved company_n_genres_output.csv (288 titles processed)